In [ ]:
import pandas as pd
import numpy as np
from onekey_algo import get_param_in_cwd
from onekey_algo.custom.components.metrics import get_time_dependent_gt
from onekey_algo.custom.components.comp1 import normalize_df

joinit_info = pd.read_csv('results/joinit_info.csv', dtype={'ID':str})
# joinit_info = normalize_df(joinit_info, not_norm=['ID', get_param_in_cwd('event_col'), get_param_in_cwd('duration_col'), 'group'], method='minmax')
joinit_info

In [ ]:
import matplotlib.pyplot as plt
from onekey_algo.custom.components.comp1 import merge_results, draw_roc
from onekey_algo.custom.components.metrics import analysis_pred_binary
from onekey_algo.custom.components.comp1 import draw_calibration
from onekey_algo.custom.components.comp1 import plot_DCA
from onekey_algo.custom.components.delong import delong_roc_test
from onekey_algo.custom.components.comp1 import draw_matrix
from onekey_algo.custom.components import stats

metric = []
hosmer = []
youden = {}
mall = []
sname = 'Sur'
event_col = get_param_in_cwd('event_col')
duration_col = get_param_in_cwd('duration_col')
model_names = [mn for mn in get_param_in_cwd('summary_models')]

# for time, tname in [(1*12, '1 Years'), (2*12, '2 Years'), (3*12, '3 Years'), (4*12, '4 Years'), (5*12, '5 Years')]:
for time, tname in [(1*12, '1 Year'), (2*12, '2 Years'),(3*12, '3 Years')]:
    gts = []
    preds = []
    for subset in get_param_in_cwd('subsets'):
        data = joinit_info[joinit_info['group'] == subset]
        time_label_data = get_time_dependent_gt(data, time=time, event_col=event_col, duration_col=duration_col)
        if len(np.unique(time_label_data['label'])) == 1:
            continue
        time_label_data['ID'] = time_label_data['ID'].astype(str)
        ALL_results = merge_results(data, time_label_data)
        ALL_results = ALL_results.drop_duplicates('ID')
        print(tname, subset, ALL_results.shape)
        ALL_results.to_csv(f'results/{sname}_time_roc_{tname}_{subset}.csv', index=False)
        gt = [np.array(ALL_results['label']) for _ in model_names]
        pred_train = [np.clip(np.array(ALL_results[d]), 0, 1000) for d in model_names]
        gts.extend(gt)
        preds.extend(pred_train)
        draw_roc(gt, pred_train, labels=model_names, title=f'Cohort {subset} {tname} ROC')
        plt.savefig(f'img/{tname}_{sname}_{subset}_auc.svg')
        plt.show()
        
        
        # Plot DeLong test results
        delong = []
        delong_columns = []
        this_delong = []
        plt.figure(figsize=(12, 10))
        cm = np.zeros((len(model_names), len(model_names)))
        for i, mni in enumerate(model_names):
            for j, mnj in enumerate(model_names):
                if i <= j:
                    cm[i][j] = np.nan
                else:
                    cm[i][j] = delong_roc_test(ALL_results['label'], ALL_results[mni], ALL_results[mnj])[0][0]
        cm = pd.DataFrame(cm[1:, :-1], index=model_names[1:], columns=model_names[:-1])
        draw_matrix(cm, annot=True, cmap='jet_r', cbar=True)
        plt.title(f'Cohort {subset} Delong')
        plt.savefig(f'img/delong_each_cohort_{tname}_{subset}.svg', bbox_inches = 'tight')
        plt.show()
        
        plot_DCA(pred_train, gt[0], title=f'Cohort {subset} {tname} DCA', labels=model_names, y_min=-0.15, remap=True)
        plt.savefig(f'img/{tname}_{sname}_{subset}_dca.svg')
        plt.show()
        
        draw_calibration(pred_scores=pred_train, n_bins=5, remap=True, y_test=gt, model_names=model_names, add_0=True)
        plt.title(f"Cohort {subset} {tname} Calibration")
        plt.savefig(f'img/{tname}_{sname}_{subset}_cali.svg')
        plt.show()
        
        hosmer.append([stats.hosmer_lemeshow_test(y_true, y_pred, bins=25, remap=False) 
                  for fn, y_true, y_pred in zip(model_names, gt, pred_train)] + [tname, subset])
        
        for mname, y, score in zip(model_names, gt, pred_train):
            # Calculate evaluation cohort metrics
            acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y, score)
            ci = f"{ci[0]:.4f} - {ci[1]:.4f}"
            metric.append((mname, acc, auc, ci, tpr, tnr, ppv, npv, tname, f"{subset[0].upper() + subset[1:]}"))
            youden[mname] = thres
            
    for idx, mn in enumerate(model_names):
        draw_roc(gts[idx::len(model_names)], preds[idx::len(model_names)], 
                 labels=get_param_in_cwd('subsets'), title=f"{mn} {tname} Time Dependent ROC")
        plt.savefig(f'img/{tname}_{sname}_{mn}_auc.svg')
        plt.show()

pd.set_option('display.max_rows', None)
pd.DataFrame(metric, index=None, columns=['Signature', 'Accuracy', 'AUC', '95% CI', 'Sensitivity', 'Specificity', 
                                          'PPV', 'NPV', 'Survival', 'Cohort'])